In [ ]:
#@title 🔌 [اختیاری] اتصال MCP Agent — کنترل Colab از طریق Antigravity
#@markdown ### ℹ️ برای کنترل این Notebook از Antigravity MCP:
#@markdown 1. در Antigravity ابزار `open_colab_browser_connection` را اجرا کنید.
#@markdown 2. سپس همین سلول را اجرا کنید — اتصال خودکار برقرار می‌شود.
#@markdown 3. وضعیت اتصال را در **Console مرورگر (F12)** ببینید.

from IPython.display import Javascript, display

js_code = '''
(function() {
  const hash = window.location.hash || '';\n  const params = new URLSearchParams(hash.startsWith('#') ? hash.slice(1) : hash);\n  const token = params.get('mcpProxyToken');\n  const port  = params.get('mcpProxyPort');\n  const box = document.createElement('div');
  box.style = 'padding:12px;border-radius:8px;font-family:sans-serif;font-size:14px;margin:8px 0;border:1px solid #ccc';
  function setStatus(msg, bg) { box.style.background=bg; box.innerHTML=msg; console.log('[MCP]',msg.replace(/<[^>]+>/g,'')); }
  var el = document.querySelector('#mcp-status-box');
  if (el) el.remove();
  box.id = 'mcp-status-box';
  document.body.appendChild(box);
  if (!token || !port) {
    setStatus('⚠️ <b>mcpProxyToken در URL نیست.</b> در Antigravity open_colab_browser_connection را اجرا کرده، صفحه را refresh و این سلول را دوباره اجرا کنید.', '#fff3e0');\n    return;
  }
  setStatus('⏳ در حال اتصال به MCP Agent (پورت ' + port + ')...', '#e3f2fd');\n  try {
    var ws = new WebSocket('ws://127.0.0.1:' + port + '/?access_token=' + encodeURIComponent(token), ['mcp']);\n    ws.onopen  = function() { setStatus('✅ <b>اتصال MCP برقرار شد!</b> Antigravity می‌تواند این Notebook را کنترل کند.', '#e8f5e9'); };\n    ws.onerror = function() { setStatus('❌ <b>خطا در اتصال.</b> مطمئن شوید open_colab_browser_connection در Antigravity در حال اجراست.', '#ffebee'); };\n    ws.onclose = function(e) { if (e.code!==1000) setStatus('⚠️ قطع شد (کد: '+e.code+')', '#fff3e0'); };\n    window._mcpWs = ws;
  } catch(err) { setStatus('❌ خطا: ' + err.message, '#ffebee'); }
})();
'''
display(Javascript(js_code))
print('✅ MCP Connector اجرا شد — وضعیت را در output بالا ببینید.')


#@title 🟢 [روش اول] راه‌اندازی ۱۰۰٪ تک‌کلیک با KasmVNC مدرن (Live Real-Time Logs)
#@markdown ### 🌐 تنظیمات آدرس پروژه و رزولوشن صفحه:
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1366x768", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re, shutil
from IPython.display import display, HTML

def run_step(step_num, title, cmd, show_output=False):
    print(f"\n▶ [{step_num}/6] {title}...", flush=True)
    t0 = time.time()
    if show_output:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in iter(proc.stdout.readline, ''):
            l = line.strip()
            if l: print(f"  │ {l[:110]}", flush=True)
        proc.wait()
        code = proc.returncode
    else:
        code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# 1. سخت‌افزار کرنل، لودر LunarG Vulkan 1.4 و ICD انویدیا
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس، لودر Vulkan و درایور سخت‌افزاری NVIDIA T4", """
os_mknod() {
  mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d /etc/apt/sources.list.d
  mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
  mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
  mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
  chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true

  wget -qO - https://packages.lunarg.com/lunarg-signing-key-pub.asc | apt-key add - >/dev/null 2>&1 || true
  wget -qO /etc/apt/sources.list.d/lunarg-vulkan.list https://packages.lunarg.com/vulkan/lunarg-vulkan-jammy.list >/dev/null 2>&1 || true
  apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools dbus-x11 >/dev/null 2>&1

  cat << 'EOF' > /etc/vulkan/icd.d/nvidia_icd.json
{
  "file_format_version": "1.0.0",
  "ICD": {
    "library_path": "libGLX_nvidia.so.0",
    "api_version": "1.3.275"
  }
}
EOF
  cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
  echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
  ldconfig
  /etc/init.d/dbus start >/dev/null 2>&1 || true
}
os_mknod
""")

# 2. کاربر اختصاصی
run_step(2, "ساخت کاربر اختصاصی colab و تنظیم مجوزها", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab >/dev/null 2>&1 || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# 3. نصب کروم
if not os.path.exists("/usr/bin/google-chrome"):
    run_step(3, "دانلود و نصب رسمی Google Chrome Stable", """
    wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f /tmp/chrome.deb
    """)
else:
    print("\n▶ [3/6] بررسی مرورگر: Google Chrome از قبل نصب است.", flush=True)

# 4. نصب KasmVNC 1.5.0 و کانفیگ HTTP بدون خطای SSL
res = subprocess.run("lsb_release -cs", shell=True, capture_output=True, text=True)
distro = res.stdout.strip() if res.stdout.strip() in ["jammy", "noble", "focal"] else "jammy"
kasm_deb = f"/tmp/kasmvnc_{distro}.deb"

if not os.path.exists("/usr/bin/vncserver") or not os.path.exists("/usr/share/kasmvnc"):
    run_step(4, f"نصب KasmVNC 1.5.0 ({distro}) و محیط XFCE4 با فشرده‌سازی WebP", f"""
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 xfce4-terminal desktop-base xauth ssl-cert >/dev/null 2>&1
    wget -q -O {kasm_deb} https://github.com/kasmtech/KasmVNC/releases/download/v1.5.0/kasmvncserver_{distro}_1.5.0_amd64.deb
    dpkg -i {kasm_deb} >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f {kasm_deb}
    adduser colab ssl-cert >/dev/null 2>&1 || true
    """)
else:
    print("\n▶ [4/6] بررسی KasmVNC: پکیج‌ها از قبل نصب هستند.", flush=True)

# کانفیگ تمیز kasmvnc.yaml برای HTTP خالص و بدون باگ گواهی SSL
os.makedirs("/home/colab/.vnc", exist_ok=True)
os.makedirs("/etc/kasmvnc", exist_ok=True)
kasm_yaml = """
network:
  protocol: http
  interface: 0.0.0.0
  websocket_port: 8443
  use_ipv4: true
  use_ipv6: false
  ssl:
    require_ssl: false
"""
with open("/etc/kasmvnc/kasmvnc.yaml", "w") as f: f.write(kasm_yaml)
with open("/home/colab/.vnc/kasmvnc.yaml", "w") as f: f.write(kasm_yaml)

# ساخت xstartup پایدار با شتاب Dawn WebGPU
xstartup = f"""#!/bin/bash
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json

(
  sleep 4
  google-chrome-stable \\
    --no-sandbox \\
    --disable-gpu-sandbox \\
    --enable-unsafe-webgpu \\
    --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
    --use-angle=vulkan \\
    --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist \\
    --disable-dawn-features=disallow_unsafe_apis \\
    --ignore-gpu-blocklist \\
    --disable-dev-shm-usage \\
    --enable-gpu-rasterization \\
    --enable-zero-copy \\
    --start-maximized \\
    "{TARGET_URL}" &
) &

exec startxfce4
"""
with open("/home/colab/.vnc/xstartup", "w") as f: f.write(xstartup)
subprocess.run("chmod +x /home/colab/.vnc/xstartup", shell=True)

# ثبت پسورد برای کاربر colab
subprocess.run("echo -e '123456\n123456\n' | sudo -u colab kasmvncpasswd -u colab -wo >/dev/null 2>&1 || true", shell=True)
subprocess.run("chown -R colab:colab /home/colab", shell=True)

# 5. تونل Cloudflare
if not os.path.exists("/usr/local/bin/cloudflared"):
    run_step(5, "دانلود باینری تونل Cloudflare", """
    wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x /usr/local/bin/cloudflared
    """)
else:
    print("\n▶ [5/6] بررسی تونل: Cloudflared آماده است.", flush=True)

# 6. استارت سرور VNC و بررسی پورت واقعی
print("\n▶ [6/6] استارت سرور KasmVNC و بازرسی سلامت ارتباط...", flush=True)
subprocess.run("sudo -u colab vncserver -kill :1 >/dev/null 2>&1 || true", shell=True)
subprocess.run("pkill -9 -f 'cloudflared' >/dev/null 2>&1 || true", shell=True)
subprocess.run("rm -rf /tmp/.X1-lock /tmp/.X11-unix/X1", shell=True)
time.sleep(1)

# استارت سرور
vnc_proc = subprocess.run(
    f"sudo -u colab vncserver :1 -geometry {RESOLUTION} -depth 24 -disableBasicAuth",
    shell=True, capture_output=True, text=True
)

# بررسی پورت باز شده (8443 یا 8444)
vnc_port = None
for _ in range(12):
    time.sleep(1)
    chk = subprocess.run("ss -tlnp 2>/dev/null || netstat -tlnp 2>/dev/null", shell=True, capture_output=True, text=True)
    if ":8443" in chk.stdout:
        vnc_port = 8443
        break
    elif ":8444" in chk.stdout:
        vnc_port = 8444
        break

if not vnc_port:
    print("❌ خطا: سرور VNC روی پورت 8443/8444 فعال نشد!")
    print("خروجی vncserver:", vnc_proc.stdout)
    if vnc_proc.stderr: print("خطای vncserver:", vnc_proc.stderr)
    print("\nآخرین خطوط لاگ VNC:")
    subprocess.run("cat /home/colab/.vnc/*.log 2>/dev/null | tail -n 25", shell=True)
    raise RuntimeError("VNC Server failed to bind to port.")

print(f"  ✔ سرور KasmVNC روی پورت داخلی {vnc_port} با موفقیت مستقر شد.", flush=True)

# استارت تونل روی پورت تایید شده با پروتکل HTTP خالص (بدون خطای 502)
log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file): os.remove(log_file)
subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{vnc_port}", "--logfile", log_file])

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print("\n" + "═"*65)
    print("🎉 محیط ابری با موفقیت لود شد! (شتاب سخت‌افزاری Tesla T4 فعال است)")
    print(f"🎯 پورت لوکال متصل‌شده: {vnc_port} | مقصد: {TARGET_URL}")
    print("═"*65)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #00e676;">⚡ دسکتاپ KasmVNC آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                ارتباط بدون خطای 502 برقرار شد. برای ورود به محیط روی دکمه زیر کلیک کنید:
            </p>
            <a href="{tunnel_url}" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به مرورگر ابری (KasmVNC)
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                لینک مستقیم: <a href="{tunnel_url}" target="_blank" style="color: #64ffda;">{tunnel_url}</a>
            </p>
            <p style="margin-top: 8px; font-size: 12px; opacity: 0.7;">
                (نام کاربری: <b>colab</b> | رمز عبور: <b>123456</b> در صورت درخواست لاگین)
            </p>
        </div>
    '''))
else:
    print("❌ خطا در ایجاد آدرس تونل Cloudflare.")


In [ ]:
#@title ⚡ [روش دوم] راه‌اندازی با Google Chrome Remote Desktop (Live Real-Time Logs)
AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}

import os, sys, time, subprocess, re
from IPython.display import display, HTML

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 دریافت کد لینوکس: https://remotedesktop.google.com/headless")
    raw_auth = input("دستور احراز هویت را وارد کنید: ").strip()

if not raw_auth:
    raise ValueError("دستور احراز هویت الزامی است.")

def run_step(step_num, title, cmd):
    print(f"\n▶ [{step_num}/5] {title}...", flush=True)
    t0 = time.time()
    code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# 1. درایورهای گرافیک و ولکان
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس، لودر ولکان و درایور تسلا T4", """
mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true

wget -qO - https://packages.lunarg.com/lunarg-signing-key-pub.asc | apt-key add - >/dev/null 2>&1 || true
wget -qO /etc/apt/sources.list.d/lunarg-vulkan.list https://packages.lunarg.com/vulkan/lunarg-vulkan-jammy.list >/dev/null 2>&1 || true
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools dbus-x11 >/dev/null 2>&1

cat << 'EOF' > /etc/vulkan/icd.d/nvidia_icd.json
{
  "file_format_version": "1.0.0",
  "ICD": {
    "library_path": "libGLX_nvidia.so.0",
    "api_version": "1.3.275"
  }
}
EOF
cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
ldconfig
/etc/init.d/dbus start >/dev/null 2>&1 || true
""")

# 2. کاربر اختصاصی
run_step(2, "ساخت کاربر اختصاصی colab و دسترسی‌های سیستمی", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab >/dev/null 2>&1 || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# 3. پکیج‌های XFCE و Chrome Remote Desktop
run_step(3, "نصب پکیج‌های دسکتاپ XFCE4 و Chrome Remote Desktop", """
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal xscreensaver >/dev/null 2>&1
if [ ! -f /opt/google/chrome-remote-desktop/start-host ]; then
  wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
  dpkg -i /tmp/crd.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/crd.deb
fi
usermod -aG chrome-remote-desktop colab >/dev/null 2>&1 || true
echo 'exec /usr/bin/xfce4-session' > /etc/chrome-remote-desktop-session
echo 'exec /usr/bin/xfce4-session' > /home/colab/.chrome-remote-desktop-session
chmod +x /home/colab/.chrome-remote-desktop-session
""")

# 4. کروم و اسکریپت WebGPU در Startup دسکتاپ
run_step(4, "نصب Google Chrome و تنظیم اجرای خودکار با شتاب Dawn WebGPU", f"""
if [ ! -f /usr/bin/google-chrome ]; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/chrome.deb
fi
mkdir -p /home/colab/.config/autostart /home/colab/Desktop
cat << 'EOF' > /home/colab/launch_chrome.sh
#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-angle=vulkan \\
  --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist \\
  --disable-dawn-features=disallow_unsafe_apis \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --start-maximized \\
  "{TARGET_URL}" &
EOF
chmod +x /home/colab/launch_chrome.sh
cat << 'EOF' > /home/colab/.config/autostart/webgpu-chrome.desktop
[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Terminal=false
EOF
cp /home/colab/.config/autostart/webgpu-chrome.desktop /home/colab/Desktop/
chmod +x /home/colab/Desktop/webgpu-chrome.desktop
chown -R colab:colab /home/colab
""")

# 5. استارت هاست با هندل کردن عدم وجود systemd در کانتینر
print("\n▶ [5/5] ثبت و راه‌اندازی هاست ریموت دسکتاپ گوگل...", flush=True)
clean_cmd = re.sub(r'^DISPLAY=\s*', '', raw_auth)
clean_cmd = re.sub(r'--pin=\S+', '', clean_cmd).strip()
clean_cmd += f" --pin={PIN}"

runner_script = f"""#!/bin/bash
{clean_cmd} || true
/opt/google/chrome-remote-desktop/chrome-remote-desktop --start || true
"""
with open("/tmp/start_crd.sh", "w") as f: f.write(runner_script)
subprocess.run("chmod +x /tmp/start_crd.sh && chown colab:colab /tmp/start_crd.sh", shell=True)

res = subprocess.run("su - colab -c /tmp/start_crd.sh", shell=True, capture_output=True, text=True)
time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop", shell=True, capture_output=True, text=True)

print("\n" + "═"*65)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Google Chrome Remote Desktop با موفقیت استارت شد!")
    print(f"🔑 پین اتصال: {PIN}")
    print("═"*65)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری ۶۰ فریم آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                دستگاه در پنل ریموت گوگل ثبت شد. روی دکمه زیر کلیک کرده و پین <b>{PIN}</b> را وارد کنید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    '''))
else:
    print("⚠️ خروجی فراخوانی:")
    print(res.stdout)
    if res.stderr: print(res.stderr)


In [ ]:
#@title 📊 مانیتورینگ زنده کارت گرافیک و توان پردازشی تسلا T4 (Universal Web3 GPU Monitor)
import time, subprocess, shutil
from IPython.display import clear_output

if not shutil.which("nvidia-smi"):
    print("❌ خطا: کارت گرافیک شناسایی نشد!")
    print("👈 لطفاً از منوی بالای کولب مسیر زیر را دنبال کرده و نوت‌بوک را روی T4 GPU قرار دهید:")
    print("   Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save")
else:
    print("🚀 مانیتورینگ زنده توان پردازشی تسلا T4 در حال اجراست (برای توقف، دکمه Stop سلول را بزنید)...")
    try:
        while True:
            res = subprocess.run([
                "nvidia-smi",
                "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
                "--format=csv,noheader,nounits"
            ], capture_output=True, text=True)
            
            if res.returncode == 0 and res.stdout.strip():
                parts = [p.strip() for p in res.stdout.strip().split(",")]
                if len(parts) >= 7:
                    name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
                    power_str = f"{power} W" if power != "[Not Supported]" else "فعال (کلاک P0)"
                    clear_output(wait=True)
                    print("="*65)
                    print(f"🚀 UNIVERSAL WEBGPU MINING & COMPUTE MONITOR — {name}")
                    print("="*65)
                    print(f"  ⚡ GPU Compute Utilization : {gpu_util}%")
                    print(f"  🧠 VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
                    print(f"  🌡️  GPU Temperature         : {temp} °C")
                    print(f"  🔋 Power Consumption       : {power_str}")
                    print("="*65)
                    print("  💡 برای متوقف کردن مانیتور، دکمه Stop (■) سلول را بزنید.")
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nمانیتور با موفقیت متوقف شد.")


In [ ]:
#@title 📊 مانیتورینگ زنده کارت گرافیک و توان پردازشی تسلا T4 (Universal Web3 GPU Monitor)
#@markdown این سلول وضعیت زنده درصد لود پردازشی GPU، حافظه VRAM، توان مصرفی وات، دما و پروسه‌های فعال را برای هر پروژه ماینینگ نمایش می‌دهد:

import time, subprocess
from IPython.display import clear_output

print("Starting Universal GPU Monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*65)
            print(f"🚀 UNIVERSAL WEBGPU MINING & COMPUTE MONITOR — {name}")
            print("="*65)
            print(f"  ⚡ GPU Compute Utilization : {gpu_util}%")
            print(f"  🧠 VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  🌡️  GPU Temperature         : {temp} °C")
            print(f"  🔋 Power Consumption       : {power} W")
            print("="*65)
            print("  💡 Press the Stop button (■) in notebook to halt the monitor.")
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
